In [1]:
import pandas as pd
pd.set_option('display.max_rows', None)

import os
from PIL import Image

In [2]:
root_path = "/Users/jwatts/astrophotography/Processed/"
output_path = "/Users/jwatts/astrophotography/Optimized"

In [3]:
def create_image_dataframe(root_path):
    data = []
    for root, dirs, files in os.walk(root_path):
        for file in files:
            if file.endswith('.png'):
                image_path = os.path.join(root, file)
                directory_name = os.path.basename(root).lower().replace(' ', '')
                data.append([image_path, directory_name])
    return pd.DataFrame(data, columns=['image_name', 'output_filename'])

def remap_filenames(df):
    remap_dict = {
        'ldn1625': 'caldwell49',
        'ngc869': 'caldwell14',
        'ngc2362': 'caldwell64'
    }

    def remap_filename(filename):
        return remap_dict.get(filename, filename)

    df['output_filename'] = df['output_filename'].apply(remap_filename)
    return df

def process_images(df, output_path):
    for index, row in df.iterrows():
        image_path = row['image_name']
        output_filename = row['output_filename']
        crop_height = row['crop']
        resize_height = row['resize']
        crop_x = row['crop_x']
        crop_y = row['crop_y']

        with Image.open(image_path) as img:
            # Determine the cropping coordinates
            if crop_height > 0:
                aspect_ratio = img.width / img.height
                new_width = int(crop_height * aspect_ratio)

                if crop_x > 0 and crop_y > 0:
                    # Crop around the specified center point
                    left = max(crop_x - new_width // 2, 0)
                    top = max(crop_y - crop_height // 2, 0)
                    right = min(crop_x + new_width // 2, img.width)
                    bottom = min(crop_y + crop_height // 2, img.height)
                else:
                    # Crop from the center of the image
                    left = (img.width - new_width) // 2
                    top = (img.height - crop_height) // 2
                    right = left + new_width
                    bottom = top + crop_height
                    
                #print(f"left:{left}, top:{top}, right:{right}, bottom:{bottom}")
                img = img.crop((left, top, right, bottom))

            # Resize the image if specified and necessary
            if resize_height > 0 and img.height > resize_height:
                aspect_ratio = img.width / img.height
                new_width = int(resize_height * aspect_ratio)
                img = img.resize((new_width, resize_height), Image.Resampling.LANCZOS)

            # Save the processed image in WEBP format in the output folder
            optimized_image_path = os.path.join(output_path, f'{output_filename}.webp')
            img.save(optimized_image_path, 'WEBP')


In [10]:
df = create_image_dataframe(root_path)
df = remap_filenames(df)
df['crop'] = 0
df['crop_x'] = 0
df['crop_y'] = 0
df['resize'] = 1080


df.loc[df['output_filename'] == "m1", 'crop'] = 1300  # Crab Nebula
df.loc[df['output_filename'] == "m13", 'crop'] = 1750  # Hercules Globular Cluster
df.loc[df['output_filename'] == "m51", 'crop'] = 1080  # Whirlpool Galaxy
df.loc[df['output_filename'] == "m81", 'crop'] = 1080  # Bode's Galaxy
df.loc[df['output_filename'] == "m81", 'crop_x'] = 760  
df.loc[df['output_filename'] == "m81", 'crop_y'] = 1600  
df.loc[df['output_filename'] == "m97", 'crop'] = 1080  # Owl nebula
df.loc[df['output_filename'] == "m101", 'crop'] = 1500  # Pinwheel Galaxy
df.loc[df['output_filename'] == "m104", 'crop'] = 1080  # Sombrero Galaxy

df[0:100]

,image_name,output_filename,crop,crop_x,crop_y,resize
0,/Users/jwatts/astrophotography/Processed/NGC 8...,caldwell14,0,0,0,1080
1,/Users/jwatts/astrophotography/Processed/M1/M1...,m1,1300,0,0,1080
2,/Users/jwatts/astrophotography/Processed/M31/M...,m31,0,0,0,1080
3,/Users/jwatts/astrophotography/Processed/M97/M...,m97,1080,0,0,1080
4,/Users/jwatts/astrophotography/Processed/M101/...,m101,1500,0,0,1080
5,/Users/jwatts/astrophotography/Processed/M78/M...,m78,0,0,0,1080
6,/Users/jwatts/astrophotography/Processed/M13/M...,m13,1750,0,0,1080
7,/Users/jwatts/astrophotography/Processed/LDN 1...,caldwell49,0,0,0,1080
8,/Users/jwatts/astrophotography/Processed/NGC 2...,caldwell64,0,0,0,1080
9,/Users/jwatts/astrophotography/Processed/M104/...,m104,1080,0,0,1080


In [11]:
# Create the 'optimized' directory if it doesn't exist
if not os.path.exists(output_path):
    os.makedirs(output_path)

process_images(df, output_path)